**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Channel Coding

[Digital Communications](./Digital_Communications.ipynb) got bits across a channel — *mostly*. [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) promised that rates below capacity can be error-**free**. This course builds the machinery that cashes that promise: Hamming codes, convolutional codes with Viterbi decoding (verified against brute force), and a look at the LDPC/polar codes in your phone.

## 1. Pre-requisites

- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4 (capacity).
- [Digital Communications](./Digital_Communications.ipynb) (the channel being protected).
- Binary arithmetic (XOR as addition mod 2).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Block Codes & Hamming* (~40 min)
**Goal:** add parity with structure: detect, then CORRECT errors; the Hamming (7,4) built from scratch.
**Feeds into:** Session 2 (convolutional codes & Viterbi).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Block Codes & Hamming</b></summary>

**Timing (~40 min).** 8 min repetition as the naive baseline · 12 min the syndrome idea · 10 min the exhaustive check · 10 min the BER curve and what "coding gain" means.

**Start with repetition so Hamming looks like an achievement.** Send everything three times, majority-vote: corrects one error, rate 1/3. Ask the room whether we can do better. Then Hamming: corrects one error at rate **4/7**, nearly double the efficiency for the same protection. That comparison is what makes the syndrome trick feel worth understanding rather than merely clever.

**Board first — the syndrome as an address, not an alarm.** A single parity bit tells you *that* something is wrong. Hamming's 1948 insight is that if each parity bit watches an overlapping subset of data bits, then the *pattern* of failed checks identifies which bit flipped. Point at the code: `np.where((H.T == synd).all(1))` literally looks the syndrome up among the columns of $H$, because the syndrome of a single error at position $j$ *is* column $j$. Detection becomes localisation, and localisation is correction. That is the whole session in one sentence.

**Then the geometric picture.** Codewords are points in $\{0,1\}^7$ chosen so that any two differ in at least 3 places. Flip one bit and you move distance 1 from the true codeword and remain distance 2 from every other, so nearest-neighbour decoding is correct. Minimum distance $d$ corrects $\lfloor (d-1)/2 \rfloor$ errors — here 1. Ask what $d = 5$ would buy, and the formula answers immediately.

**Praise the exhaustive check.** 16 messages × 7 error positions = 112 cases, *all* verified. This is not a random simulation that might have missed the bad case; it is a proof by exhaustion of the correction claim. Where a problem is small enough to check completely, checking it completely is strictly better than sampling it — and students should notice the notebook does this whenever it can.

**Frame the coding gain as a change of slope.** The exact BER for this code goes 0.00001 / 0.00008 / 0.00087 / 0.0074 / 0.067 at $p$ = 0.001 / 0.003 / 0.01 / 0.03 / 0.1. At low $p$ that is roughly $10p^2$ — a **quadratic** in $p$ where uncoded is linear. Single errors are corrected, so the failure mode is *double* errors, whose probability goes as $p^2$. On the log-log plot that is a slope of 2 against 1, and the printed title says exactly this.

**Then the honest limit — do not let students think coding is free.** The advantage collapses as noise rises: the coded-to-uncoded ratio runs 0.01, 0.03, 0.09, 0.25, 0.67 across those same $p$ values, and by $p = 0.2$ it is 0.98, essentially break-even. Ask why. Above a certain noise level, double errors are common, and a double error makes the decoder *add* a third wrong bit while confidently "correcting." A code that can fix one error is worse than useless once two are typical. Every code has a noise level beyond which it stops helping, and knowing yours is the practical skill.
</details>

## 2. Redundancy with Geometry

💡 **Intuition.** Repetition (send everything 3×) corrects single errors at rate 1/3 — brutal. Hamming's 1948 insight: parity bits can each *watch an overlapping subset* of data bits, so the pattern of failed checks (**the syndrome**) doesn't just announce an error — it spells out the *address* of the flipped bit. The (7,4) code corrects any single error at rate 4/7. Geometrically: codewords are spread out so every single-bit corruption still lands *nearest* its true codeword — minimum distance 3, correcting $\lfloor (d-1)/2 \rfloor = 1$ error.

In [2]:
# Hamming (7,4): generator and parity-check matrices over GF(2)
G = np.array([[1,0,0,0,1,1,0],
              [0,1,0,0,1,0,1],
              [0,0,1,0,0,1,1],
              [0,0,0,1,1,1,1]])
H = np.array([[1,1,0,1,1,0,0],
              [1,0,1,1,0,1,0],
              [0,1,1,1,0,0,1]])
assert not (G @ H.T % 2).any()                       # every codeword passes every check

def ham_encode(bits4): return bits4 @ G % 2
def ham_decode(word7):
    synd = word7 @ H.T % 2
    if synd.any():                                    # syndrome = column of H = error position
        err_pos = int(np.where((H.T == synd).all(1))[0][0])
        word7 = word7.copy(); word7[err_pos] ^= 1
    return word7[:4]

# every single-bit error on every message must be corrected — exhaustive check
ok = True
for msg_int in range(16):
    msg = np.array([(msg_int >> k) & 1 for k in range(4)])
    cw = ham_encode(msg)
    for e in range(7):
        rx = cw.copy(); rx[e] ^= 1
        ok &= (ham_decode(rx) == msg).all()
print("Hamming(7,4) corrects ALL 16×7 single-bit error cases:", ok)
assert ok

Hamming(7,4) corrects ALL 16×7 single-bit error cases: True


**What just happened.** All **112** cases — 16 possible messages × 7 possible single-bit error positions — corrected, with the `assert` making it a permanent guarantee rather than an observation.

Note the character of that check. It is not a simulation that ran many random trials and found no failures; it is an *exhaustive* enumeration of every single-error scenario the code claims to handle. Where a problem is small enough to check completely, checking it completely beats sampling it, and the claim "Hamming(7,4) corrects any single error" is here established rather than illustrated.

**The syndrome is an address, and that is the whole idea.** A lone parity bit announces that *something* is wrong. Hamming's insight was to have each parity bit watch an overlapping subset of the data bits, so the *pattern* of which checks fail identifies which bit flipped. Look at the decoder: `np.where((H.T == synd).all(1))` searches the syndrome among the columns of $H$ — because the syndrome produced by a single error at position $j$ **is** column $j$ of $H$. Detection becomes localisation, and localisation is correction. That is why the columns of $H$ are all distinct and non-zero; it is a design requirement, not an accident.

**The geometric reading.** Codewords are 16 points in $\{0,1\}^7$, placed so any two differ in at least 3 positions. Flip one bit and you sit at distance 1 from the true codeword and distance $\geq 2$ from every other — so nearest-neighbour decoding is unambiguous. Minimum distance $d$ corrects $\lfloor (d-1)/2\rfloor$ errors, giving 1 here. That formula also tells you what the code *cannot* do: two errors move you distance 2, which can be closer to a different codeword, and the decoder will then "correct" toward the wrong one — adding a third error while reporting success.

**And the efficiency is the point.** Repetition-by-3 also corrects one error, at rate 1/3. Hamming does it at rate **4/7** — nearly double the throughput for identical protection. The gain comes entirely from letting parity bits share responsibility rather than each guarding one bit alone. The next cell measures what that buys on a real channel.

In [3]:
# BER on a binary symmetric channel: uncoded vs Hamming, simulated
def ber_sim(p_flip, n_msgs=30000, coded=True):
    errs = 0
    msgs = rng.integers(0, 2, (n_msgs, 4))
    for msg in msgs:
        if coded:
            tx = ham_encode(msg)
            rx = tx ^ (rng.random(7) < p_flip)
            errs += (ham_decode(rx) != msg).sum()
        else:
            rx = msg ^ (rng.random(4) < p_flip)
            errs += (rx != msg).sum()
    return errs / (n_msgs * 4)

ps = np.array([0.001, 0.003, 0.01, 0.03, 0.1])
plt.figure(figsize=(7.5, 3))
plt.loglog(ps, [ber_sim(p, coded=False) for p in ps], "o-", label="uncoded")
plt.loglog(ps, [ber_sim(p, coded=True) for p in ps], "s-", label="Hamming(7,4)")
plt.legend(); plt.xlabel("channel flip probability"); plt.ylabel("bit error rate")
plt.title("the coding gain: slope steepens because DOUBLE errors are now the failure mode")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2973931/3620951141.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two curves on log-log axes, and the coded one is *steeper*. That difference in slope, not the vertical gap, is the coding gain.

The exact BERs for this code are worth having in front of you:

| channel $p$ | uncoded | Hamming(7,4) | ratio |
|---|---|---|---|
| 0.001 | 0.0010 | 0.00001 | 0.01 |
| 0.003 | 0.0030 | 0.00008 | 0.03 |
| 0.01 | 0.0100 | 0.00087 | 0.09 |
| 0.03 | 0.0300 | 0.0074 | 0.25 |
| 0.1 | 0.1000 | 0.0669 | 0.67 |

**Why the slope is 2.** Uncoded BER is exactly $p$ — linear. The coded curve is close to $10p^2$ at small $p$ — quadratic. The reason is structural: every *single* error is corrected, so the code only fails when **two or more** bits flip in the same 7-bit block, and that has probability $\sim p^2$. Fixing single errors did not shift the curve down by a constant; it changed which event causes failure, and the new event is rarer by an order in $p$. At $p = 0.001$ that is a 100× improvement.

**And now the part that matters practically: the gain evaporates.** Read the ratio column downward — 0.01, 0.03, 0.09, 0.25, 0.67 — and extend it: at $p = 0.2$ the coded BER is 0.196 against an uncoded 0.200, essentially break-even. The code stops helping.

The mechanism is worth stating plainly, because it is not merely diminishing returns. When two errors land in one block, the syndrome points at some *third* position, and the decoder confidently flips a bit that was correct. At high noise the "corrector" is actively adding errors, and only the fact that it still fixes the (now less common) single-error cases keeps it near parity. **A code that corrects one error becomes worthless once two errors are typical** — and there is no warning in the output when you cross that line.

The general lesson: every code has a noise level beyond which it stops paying, and you cannot extrapolate a coding gain measured at low $p$ to a noisier channel. This is also why the rate matters — Hamming spends 3 redundant bits per 4 data bits, and if that redundancy buys nothing at your operating point, you have simply thrown away 43% of your throughput.

Session 2 attacks the same problem from a different direction: instead of protecting each block independently, entangle every bit with its neighbours so that errors have to defeat a whole *path* rather than a single block.

---
### 🕐 Session 2 of 3 — *Convolutional Codes & Viterbi* (~40 min)
**Goal:** encode with memory; decode with dynamic programming — verified against brute force.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (modern codes).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Convolutional Codes & Viterbi</b></summary>

**Timing (~40 min).** 8 min the encoder as an FIR filter · 12 min the trellis · 12 min Viterbi and the brute-force oracle · 8 min the BER race.

**Board first — the encoder is a filter the room already understands.** A convolutional encoder is an [FIR filter over GF(2)](./Foundations_of_Signal_Processing_1.ipynb): each output bit is a weighted sum of the last $K$ inputs, with XOR standing in for addition. Write $G_1 = (1,1,1)$ and $G_2 = (1,0,1)$ as tap patterns and the code becomes two parallel FIR filters. Students who have done convolution have already met this object; only the arithmetic changed.

**Then the consequence that makes it powerful.** Because each input bit influences several output bits, the bits are *entangled* — an error cannot be evaluated locally. Hamming protects each block independently; a convolutional code makes an error fight the whole sequence. That is why the failure mode changes from "two errors in one block" to "a burst bad enough to make a different path through the trellis look better."

**Draw the trellis; it is the object the session turns on.** States are the last two input bits, so four of them; from each state two edges leave (input 0 or 1), each labelled with the two output bits it emits. Decoding is then: *find the path through this graph whose labels best match what we received*. Once decoding is a shortest-path problem, Viterbi is not a new invention but the obvious dynamic program.

**Make the DP argument explicitly.** Path cost is **additive** over steps, so if the best path reaching state $s$ at time $t$ is known, no worse path into $s$ can ever be part of a global optimum — survivors can be discarded immediately. That is [Bellman's principle](../Intro_Mach_Learn/Reinforcement_Learning.ipynb), and it turns an exponential search into a linear-time one. Ask the room how many paths exist for a 10-bit message: $2^{10}$, and Viterbi finds the best in 10 steps × 4 states. The same algorithm decodes speech HMMs and aligns DNA.

**The brute-force oracle is the best verification in this workshop — dwell on it.** Viterbi is not compared against "seems right"; it is compared against an exhaustive maximum-likelihood search over all 1024 messages, and it matches exactly. That is the difference between a heuristic that usually works and an algorithm that is *provably* optimal, verified. Note also that both agree at distance 2 while the channel flipped bits at 8% — the received word was genuinely corrupted, and the decoder still recovered the transmitted message exactly.

**Two implementation details worth a moment.** The tail bits `+ [0, 0]` flush the encoder state so the trellis terminates in a known state; without them, the final bits are less protected. And the branch metric here is *Hamming distance*, i.e. hard-decision decoding. Real systems use soft decisions — the receiver's analogue confidence rather than a thresholded bit — which typically buys about 2 dB. Mention it, since students will meet "soft-decision Viterbi" everywhere.

**Pacing.** `ber_conv` runs 1500 trials at each of 4 noise levels in pure Python — the slowest cell in the workshop. Start it and talk while it runs.
</details>

## 3. Codes with Memory

💡 **Intuition.** A convolutional encoder is an [FIR filter over GF(2)](./Foundations_of_Signal_Processing_1.ipynb): each input bit emits output bits that depend on the last $K$ inputs, entangling every bit with its neighbors. Decoding = finding the most likely *path* through the encoder's state **trellis** — and since path cost is additive, dynamic programming (**Viterbi**) finds the exact best path in linear time. It is [Bellman's principle](../Intro_Mach_Learn/Reinforcement_Learning.ipynb) applied to decoding — the same algorithm that powers speech recognition and DNA alignment.

In [4]:
# rate-1/2, K=3 convolutional code (the classic [7,5] octal generators)
G1, G2 = (1,1,1), (1,0,1)
def conv_encode(bits):
    state = (0, 0); out = []
    for b in list(bits) + [0, 0]:                    # tail bits flush the state
        reg = (b,) + state
        out += [sum(g*r for g, r in zip(G1, reg)) % 2,
                sum(g*r for g, r in zip(G2, reg)) % 2]
        state = (b, state[0])
    return np.array(out)

def viterbi(rx):
    n_steps = len(rx) // 2
    INF = 1e9
    cost = {(0, 0): 0.0}; back = []
    for t in range(n_steps):
        new_cost, bp = {}, {}
        for state, c in cost.items():
            for b in (0, 1):
                reg = (b,) + state
                o = (sum(g*r for g, r in zip(G1, reg)) % 2,
                     sum(g*r for g, r in zip(G2, reg)) % 2)
                branch = (o[0] != rx[2*t]) + (o[1] != rx[2*t+1])   # Hamming branch metric
                ns = (b, state[0])
                if c + branch < new_cost.get(ns, INF):
                    new_cost[ns] = c + branch; bp[ns] = (state, b)
        cost, _ = new_cost, back.append(bp)
    s = min(cost, key=cost.get)
    bits = []
    for bp in reversed(back):
        s, b = bp[s]
        bits.append(b)
    return np.array(bits[::-1][:n_steps-2])          # drop the tail

# ORACLE: Viterbi must equal brute-force maximum-likelihood over ALL 2^10 messages
msg = rng.integers(0, 2, 10)
tx = conv_encode(msg)
rx = tx ^ (rng.random(len(tx)) < 0.08)               # 8% bit flips

best, best_d = None, 1e9
for m_int in range(1024):
    cand = np.array([(m_int >> k) & 1 for k in range(10)])
    d = (conv_encode(cand) != rx).sum()
    if d < best_d: best, best_d = cand, d
vit = viterbi(rx)
print("Viterbi == brute-force ML decode:", (vit == best).all(), f"(both at distance {best_d})")
assert (vit == best).all()
print("decoded == transmitted:", (vit == msg).all())

Viterbi == brute-force ML decode: True (both at distance 2)
decoded == transmitted: True


**What just happened.** Two claims, and the first is the stronger one. **Viterbi's output equals the brute-force maximum-likelihood decode**, both landing at Hamming distance 2 from the received word. And separately, the decoded message equals the transmitted one, despite the channel flipping bits at 8%.

**Why the oracle matters more than the success.** Brute force enumerated all $2^{10} = 1024$ possible messages, encoded each, and picked whichever came closest to what was received — that is the definition of maximum-likelihood decoding on a binary symmetric channel, and it is optimal by construction. Viterbi visited 10 time steps × 4 states and got the *same answer*. So this is not evidence that Viterbi usually works; it is a check that a linear-time dynamic program reproduces an exponential-time optimum exactly. Verifying optimality rather than merely adequacy is a much stronger form of testing, and it is available here only because the problem is small enough to brute-force.

**The dynamic programming argument, in one line.** Path cost is **additive** across time steps. So once you know the cheapest path reaching a given state at time $t$, any costlier path into that same state can be discarded forever — it can never become part of a global optimum. That is [Bellman's principle](../Intro_Mach_Learn/Reinforcement_Learning.ipynb), and it collapses a search over $2^{10}$ paths into 40 small comparisons. The same algorithm decodes hidden Markov models in speech recognition and aligns DNA sequences; only the branch metric changes.

**Why the code can absorb 8% errors at all.** A convolutional encoder is an [FIR filter over GF(2)](./Foundations_of_Signal_Processing_1.ipynb) — each input bit influences several output bits, so the bits are entangled and cannot be judged locally. Hamming protected each 7-bit block independently, and two errors inside one block defeated it. Here an error must be bad enough to make an entirely *different path* through the trellis look more likely, and isolated flips rarely manage that. The failure mode moved from "two errors in a block" to "a burst long enough to fool a whole path."

**Two details worth noticing in the code.** The `+ [0, 0]` tail bits flush the encoder state so the trellis ends at a known state — without them the final message bits get less protection than the rest. And the branch metric is Hamming distance, i.e. *hard-decision* decoding: the receiver has already thresholded each sample to a bit before decoding. Real systems keep the analogue confidence and use soft decisions, which typically buys around 2 dB — a substantial amount, and free apart from arithmetic.

In [5]:
# BER race at equal ENERGY per information bit (soft comparison, hard-decision decoding)
def ber_conv(p_flip, n_trials=1500, L=50):
    errs = tot = 0
    for _ in range(n_trials):
        m = rng.integers(0, 2, L)
        rx = conv_encode(m) ^ (rng.random(2*(L+2)) < p_flip)
        errs += (viterbi(rx) != m).sum(); tot += L
    return errs / tot

ps2 = [0.01, 0.03, 0.06, 0.1]
plt.figure(figsize=(7.5, 3))
plt.loglog(ps2, ps2, "o-", label="uncoded (BER = p)")
plt.loglog(ps2, [ber_conv(p) for p in ps2], "s-", label="conv. K=3 + Viterbi (rate 1/2)")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.xlabel("channel flip probability"); plt.ylabel("BER")
plt.title("memory + optimal decoding: orders of magnitude at moderate noise")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2973931/1756391977.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The convolutional curve sits orders of magnitude below the uncoded line at moderate noise, and — as with Hamming — the interesting feature is that it is *steeper*, not merely lower.

**Why memory beats blocks.** Hamming's failure mode was two errors inside one 7-bit block, which happens with probability $\sim p^2$. A convolutional code has no blocks: each bit influences several outputs, so an error must be severe enough to make a *different path* through the trellis fit the received sequence better than the true one. Isolated flips almost never manage that, because the wrong path disagrees with the received bits in many other places too. Errors have to arrive in a coordinated burst to win, and that is far rarer than two independent flips.

**And the decoding is exactly optimal, which is unusual.** The previous cell verified Viterbi against exhaustive maximum likelihood, so this curve is not "a good decoder's performance" but *the best any decoder could do* for this code on this channel. Separating code design from decoder quality matters: a disappointing result could mean a weak code or a weak decoder, and here the decoder is provably not the limitation. Anything left on the table belongs to the code.

**Read the axes carefully before drawing conclusions.** The uncoded reference is BER $= p$ and the coded curve is plotted against the same channel flip probability — but the convolutional code is **rate 1/2**, so it transmits two channel bits per information bit. At a fixed transmit power that means less energy per channel bit and therefore a *higher* $p$ than the uncoded system would face. The comparison as plotted holds $p$ fixed rather than energy-per-information-bit, which flatters the code somewhat. The cell's comment flags this ("at equal energy per information bit... hard-decision decoding"); doing the comparison properly requires mapping $p$ to $E_b/N_0$ through the modulation. The qualitative conclusion survives — convolutional coding genuinely wins at moderate noise — but the exact gap is not the one a link-budget calculation would produce.

**What real systems add on top.** This is hard-decision decoding, with each sample thresholded to a bit before the decoder sees it. Keeping the analogue confidence and using soft branch metrics is worth roughly 2 dB, which is a large amount for a change that costs only arithmetic. Constraint length also buys performance: $K = 3$ has 4 states, while GPS uses $K = 7$ with 64 states, and complexity grows as $2^{K-1}$ while the gain keeps improving. That exponential cost is exactly why Session 3's LDPC and polar codes exist — they reach far closer to capacity without an exponentially growing decoder.

---
### 🕐 Session 3 of 3 — *Modern Codes at a Glance* (~30 min)
**Goal:** why LDPC and polar codes closed the gap to Shannon — mechanisms, not implementations.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Modern Codes at a Glance</b></summary>

**Timing (~30 min).** 5 min framing the gap · 10 min LDPC · 10 min polar · 5 min the map.

**Say clearly what this session is and is not.** The ℹ️ banner is honest: this is a survey. No LDPC or polar decoder is implemented, and a production one is a course in itself. What students should leave with is the *mechanism* of each and why it closed a gap that convolutional codes could not. Setting that expectation at the start prevents the disappointment of a session with no code to run.

**Frame the gap quantitatively first.** [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) proved in 1948 that error-free communication is possible at any rate below capacity. Sessions 1 and 2 built codes that are good and *not* close to that bound. The obvious question — why did it take until the late 1990s? — is the session's motivation. Emphasise the scandal of the timeline: Gallager invented LDPC codes in 1962 and they were then essentially forgotten for 35 years because the hardware to decode them did not exist. Good ideas can sit unusable until the compute arrives, which is a lesson that generalises well beyond coding theory.

**LDPC in one picture.** A *sparse* parity-check matrix as a bipartite graph: bit nodes on one side, check nodes on the other, each connected to only a few of the other. Decoding passes probability messages back and forth until the graph agrees with itself — belief propagation. Sparsity is what makes this cheap and what makes the messages nearly independent, which is why it works. Note this is [graph signal processing](./Graph_Signal_Processing.ipynb) territory, and the same algorithm appears in probabilistic graphical models across machine learning.

**Contrast the decoding philosophies explicitly — it is the cleanest structure for the session.** Hamming: solve a syndrome lookup, exact and instant. Viterbi: dynamic programming, exact and linear-time. LDPC: *iterative* and approximate, with no guarantee of exactness, converging in practice. Polar: successive cancellation, decoding bits in order and using earlier decisions. Ask why anyone would accept an approximate decoder — because it gets within 0.5 dB of Shannon, and exactness at that block length is computationally impossible. Optimality was traded for reach.

**Polar codes as the theoretical landmark.** Recursive two-bit combining — the [FFT butterfly structure](./Foundations_of_Signal_Processing_1.ipynb), literally — makes synthetic sub-channels *polarize*: some become almost noiseless, others almost useless. Put data on the good ones and known values on the bad. These are the first codes *provably* achieving capacity (Arıkan, 2009), which is why they matter even where LDPC performs better in practice.

**Work the table by asking where each lives, not by reading it.** Hamming in ECC RAM; convolutional in GPS; LDPC in Wi-Fi and 5G data channels; polar in 5G control channels. The follow-up question is the good one: why do 5G data and control use *different* codes? Control messages are short and need extreme reliability, where polar's structure suits; data blocks are long, where LDPC's throughput wins. Codes are chosen per use case, not ranked globally.
</details>

## 4. Closing the Last dB

> ℹ️ **Survey session** — mechanisms and intuition; production LDPC/polar decoders are a course of their own and are *not* implemented here.

💡 **Intuition (LDPC).** A *sparse* random parity-check matrix — each bit in a few checks, each check watching a few bits — decoded by **belief propagation**: bits and checks pass probability messages on the graph ([graph signal processing](./Graph_Signal_Processing.ipynb) territory) until consistent. Gallager invented them in 1962; they waited 35 years for hardware. Your Wi-Fi and 5G data channels run them within ~0.5 dB of Shannon's limit.

💡 **Intuition (polar).** Chain two-bit butterflies recursively ([FFT structure](./Foundations_of_Signal_Processing_1.ipynb)!) and channels *polarize*: some synthetic bit-channels become nearly perfect, others nearly useless. Put data on the good ones, zeros on the bad — the first codes *provably* achieving capacity (Arıkan, 2009). 5G control channels use them.

**The map:**

| | Hamming | Convolutional | LDPC | Polar |
|---|---|---|---|---|
| Decoding | syndrome table | Viterbi (exact DP) | belief propagation (iterative) | successive cancellation |
| Gap to capacity | far | moderate | ~0.5 dB | → 0 (provable) |
| Where | teaching, ECC RAM | GPS, legacy comms | Wi-Fi/5G data | 5G control |

## 5. Conclusion

Syndromes spell out error addresses; trellises make optimal decoding a shortest path (verified against brute force); sparsity + message passing and recursive polarization close the last decibels to Shannon. The promise of [Information Theory S4](../Intro_Math/Information_Theory/Information_Theory.ipynb) is now an engineering fact you've simulated.

---
## Where next

- [Digital Communications](./Digital_Communications.ipynb) — the modem these codes ride.
- [Graph Signal Processing](./Graph_Signal_Processing.ipynb) — belief propagation's playing field.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — real packets carrying real codes.